# BloodBridge AI — Feature Engineering

In this notebook, we construct our temporal features and set up our prediction targets:
- Group-wise sorting by blood bank and blood type.
- 7-day rolling averages of inventory ratios, donations, and requests (shifted by 1 day to prevent lookahead leakage).
- Supply-demand gaps.
- Target definition: `shortage_next_day = 1` if the next-day inventory level drops below the shortage threshold (18% of capacity).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate project root
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
    
DATA_DIR = ROOT / "data"
print(f"Project root identified: {ROOT}")

Project root identified: C:\Users\dhair\OneDrive\Desktop\College\Project\project implementation part\BloodBridge_AI


### Load the Preprocessed Dataset

In [2]:
input_path = DATA_DIR / "interim" / "preprocessed_synthetic_bloodbridge.csv"
df = pd.read_csv(input_path, parse_dates=["record_date"])
print(f"Loaded {len(df):,} rows.")

Loaded 25,920 rows.


### Generate Rolling Lag Features

We calculate rolling window metrics over the past 7 days. To prevent data leakage, we perform a `.shift(1)` before computing the rolling averages, ensuring that predicting "tomorrow's risk" only uses historical information available "today".

In [3]:
group_keys = ["bank_id", "blood_group"]
df = df.sort_values(group_keys + ["record_date"]).copy()

# Rolling calculations for requests, donations, and inventory ratio
rolling_cols = ["requests_received", "donations_received", "inventory_ratio"]
for col in rolling_cols:
    df[f"{col}_rolling_7d"] = (
        df.groupby(group_keys)[col]
        .transform(lambda s: s.shift(1).rolling(window=7, min_periods=3).mean())
    )

# Compute gap feature
df["request_donation_gap_7d"] = df["requests_received_rolling_7d"] - df["donations_received_rolling_7d"]

print("Rolling lag features computed successfully.")

Rolling lag features computed successfully.


### Define Target Label: Next-Day Shortage Risk

Our target is predicting whether a blood bank will experience a critical shortage tomorrow. 
- We shift the `inventory_ratio` backward by 1 day (`shift(-1)`) to look at the next-day state.
- A shortage occurs if the next-day inventory level drops below `0.18` (18% of storage capacity).

In [4]:
df["next_day_inventory_ratio"] = df.groupby(group_keys)["inventory_ratio"].shift(-1)
df["shortage_next_day"] = (df["next_day_inventory_ratio"] < 0.18).astype(int)

# Clean rows with missing lag or target data (e.g. edge elements of shift)
featured_df = df.dropna().copy()

shortage_rate = featured_df["shortage_next_day"].mean()
print(f"Feature matrix shape: {featured_df.shape}")
print(f"Global shortage risk rate: {shortage_rate:.2%}")

Feature matrix shape: (25728, 25)
Global shortage risk rate: 25.84%


### Export Feature Engineered Dataset

In [5]:
output_path = DATA_DIR / "processed" / "feature_engineering_working_synthetic.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
featured_df.to_csv(output_path, index=False)
print(f"Saved feature matrix to {output_path.resolve()}")

Saved feature matrix to C:\Users\dhair\OneDrive\Desktop\College\Project\project implementation part\BloodBridge_AI\data\processed\feature_engineering_working_synthetic.csv
